In [1]:
import pandas as pd
import numpy as np
from app.data.database import engine
from sqlalchemy import text
from datetime import date
from app.core.config import settings


connected successfully; version PostgreSQL 17.8 (Ubuntu 17.8-1.pgdg22.04+1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0, 64-bit


# Data  Extraction


In [ ]:
# data extraction
revenue_rep_query = text(f"""
WITH revenue_accounts AS (
    SELECT unnest(ARRAY[{settings.DB_rev_accounts}]) AS account_id
)
SELECT
    CASE WHEN ams.journal_id in ({settings.DB_van_journals}) 
        THEN 'Van_:' || ams.journal_id::text
        ELSE 'Warehouse_' || wss.warehouse_id::text
    END AS warehouse_label,
    wss.warehouse_id, wss.journal_id, 
    amsl.price_subtotal,
    CASE WHEN ams.move_type = 'out_refund'
         THEN -amsl.price_subtotal
         ELSE  amsl.price_subtotal
    END  AS revenue_signed,
    CASE WHEN ams.move_type = 'out_refund'
         THEN -amsl.quantity
         ELSE  amsl.quantity
    END AS qty_signed
 
FROM account_move_sales_line amsl
JOIN account_move_sales ams ON ams.id = amsl.move_id
JOIN warehouse_sales_scope wss 
    ON (wss.scope_type = 'journal' AND wss.journal_id = ams.journal_id)
    OR (wss.scope_type = 'company' AND wss.company_id = ams.company_id)
 
WHERE ams.state = 'posted'
  AND amsl.account_id IN (SELECT account_id FROM revenue_accounts)
  AND ams.partner_id NOT IN ({settings.DB_partners_exclude})
  AND ams.invoice_date BETWEEN :start AND :end;
"""
)

# params
params_dict = {
    "start": "2026-08-09",  
    "end": "2026-08-15",  
}

# with engine.connect().execution_options(stream_results=True) as conn:
#     for i, chunk in enumerate(pd.read_sql(ams_query, conn, params=params_dict, chunksize=2_000_000)):
#         chunk.to_parquet(f"extract/amsl_on_ams_pp/amsl_on_ams_pp{i:04d}.parquet", index=False)

# loading to df
df_amsl_on_ams = pd.read_sql(revenue_rep_query, engine, params=params_dict)

engine.dispose()



# Data review

In [3]:
df_amsl_on_ams.head()

,warehouse_label,warehouse_id,journal_id,price_subtotal,revenue_signed,qty_signed
0,Warehouse_1,1,NaN,64.49,-64.49,-1.0
1,Warehouse_1,1,NaN,39.99,-39.99,-1.0
2,Warehouse_1,1,NaN,24.99,-24.99,-1.0
3,Warehouse_16,16,NaN,123.55,-123.55,-1.0
4,Warehouse_1,1,NaN,14.49,-14.49,-1.0


# Revenue reproduction by warehouse and van

In [4]:

net_revenue_by_wh = (df_amsl_on_ams.groupby('warehouse_label', as_index=False)[['revenue_signed', 'qty_signed']].sum().sort_values(by='warehouse_label', ascending=True))

print(net_revenue_by_wh[['warehouse_label', 'revenue_signed', 'qty_signed']])

print("\n total per van and warehouse:\n")
print(
    net_revenue_by_wh.assign(
        group_type=np.where(
            net_revenue_by_wh['warehouse_label']
            .astype(str)
            .str.lower()
            .str.startswith('van'),
            'van',
            'warehouse',
        )
    )
    .groupby('group_type', as_index=False)['revenue_signed'].sum()
)



   warehouse_label  revenue_signed  qty_signed
0         Van_:351         7446.44      168.00
1         Van_:354        37856.18     1202.00
2         Van_:375        22029.17      587.00
3         Van_:379        25852.33      747.00
4         Van_:405         3804.23      160.00
5      Warehouse_1       709453.23    19629.00
6     Warehouse_10       986777.97    23295.00
7     Warehouse_12       335685.73    11333.00
8     Warehouse_14      1070497.87    45676.00
9     Warehouse_16       626736.82    18526.00
10     Warehouse_2        29097.98     1987.00
11     Warehouse_3      1063868.44    36579.49
12     Warehouse_4       769709.77    23893.00
13     Warehouse_5        89610.56     2892.60

 total per van and warehouse:

  group_type  revenue_signed
0        van        96988.35
1  warehouse      5681438.37
